# HAM10000 Day 1: audit and exploration

Attach the original HAM10000 Kaggle dataset. Run from the repository root, or set `PROJECT_ROOT` below to the folder containing `src/`. This notebook imports the shared pipeline; the command-line entry points do the same work.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image as DisplayImage

PROJECT_ROOT = Path.cwd().resolve()  # Change this if the notebook starts elsewhere.
assert (PROJECT_ROOT / 'src').is_dir(), f'Cannot find src/ under {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))
from src.prepare_data import run_preparation, verify_splits
from src.data_pipeline import run_smoke

INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').is_dir() else PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'day1_artifacts'
print('Input:', INPUT_ROOT, 'Output:', OUTPUT_DIR)

## Audit and fixed splits
The audit stops on missing files, unreadable images, duplicate IDs, missing labels, or conflicting diagnoses within a lesion. Existing complete splits are verified and reused.

In [ ]:
parts = run_preparation(input_root=INPUT_ROOT, output_dir=OUTPUT_DIR)
display(verify_splits(parts))
display(pd.read_csv(OUTPUT_DIR / 'audit_issues.csv').head())
display(pd.read_csv(OUTPUT_DIR / 'split_counts.csv'))

## Explore class balance and RGB samples
The figures below are generated by the reusable plotting function called during preparation. Compare the image and lesion counts; check that the sample images load correctly.

In [ ]:
display(DisplayImage(filename=str(OUTPUT_DIR / 'class_distribution.png')))
display(DisplayImage(filename=str(OUTPUT_DIR / 'sample_images.png')))
display(parts['train']['dx'].value_counts().rename('train images').to_frame())

## Inspect one batch and run the tiny smoke check
This trains a small random network for one batch only. It verifies finite loss and checkpoint save/reload; it is not an assignment experiment.

In [ ]:
smoke = run_smoke(output_dir=OUTPUT_DIR, batch_size=32)  # Use 16 if memory is tight.
display(DisplayImage(filename=str(OUTPUT_DIR / 'first_batch.png')))
display(pd.Series(smoke, name='smoke check').to_frame())